In [1]:
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
print("project root:", PROJECT_ROOT)

project root: C:\Users\Anshul\Agentic_RAG


In [2]:
from dotenv import load_dotenv

load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print("Tavily key set:", bool(TAVILY_API_KEY))

Tavily key set: True


In [4]:
from tavily import TavilyClient

client = TavilyClient(api_key=TAVILY_API_KEY)

response = client.search(
    query = 'Anthropic AI safety research 2024',
    search_depth='basic',
    max_results=3,
    include_answer=True
)

print("quick answer:", response.get("answer", ""))
print("\n--- raw results ---")
for r in response["results"]:
    print(r["title"])
    print(r["url"])
    print(r["content"][:200])
    print()

quick answer: Anthropic achieved a breakthrough in AI safety in late 2024. The Anthropic Fellows Program for AI safety research opened in 2026, focusing on various safety research areas. The U.S. AI Safety Institute collaborated with Anthropic for AI safety research.

--- raw results ---
Anthropic: Building Safe and Powerful AI - Case - Faculty & Research
https://www.hbs.edu/faculty/Pages/item.aspx?num=65886
In late 2024, Anthropic, a leading AI safety and research company, achieved a significant breakthrough with computer use capabilities that allowed AI to

Anthropic Fellows Program for AI safety research
https://alignment.anthropic.com/2025/anthropic-fellows-program-2026
# Anthropic Fellows Program for AI safety research: applications open for May & July 2026. The Anthropic Fellows program provides funding and Anthropic mentorship for engineers and researchers to inve

U.S. AI Safety Institute Signs Agreements Regarding AI Safety ...
https://www.nist.gov/news-events/news/2024/08/us-

In [5]:
from langchain_core.tools import tool

def format_web_results(response: dict) -> str:
    parts = []
    
    quick_answer = response.get("answer", "")
    if quick_answer:
        parts.append(f"[Quick Answer]\n{quick_answer}")
    
    for i, r in enumerate(response["results"], 1):
        parts.append(
            f"[Web Result {i} | {r['title']} | {r['url']}]\n"
            f"{r['content']}"
        )
    
    return "\n\n---\n\n".join(parts)


In [6]:
@tool
def web_search_tool(query: str) -> str:
    """
    Search the live web for current information about AI safety and related topics.
    Use this tool when the question is about:
      - Recent news, announcements or events from 2024 onwards
      - Current status of AI policy or regulations
      - Recent statements by researchers or companies
      - Anything NOT likely to be in a research paper published before 2023

    Do NOT use this for foundational concepts already covered in research papers.
    """
    response = client.search(
        query=query,
        search_depth="basic",
        max_results=5,
        include_answer=True
    )
    
    return format_web_results(response)

In [7]:
result = web_search_tool.invoke({"query": "what has Anthropic announced about AI safety in 2025?"})
print(result)

[Quick Answer]
In 2025, Anthropic announced it could not rule out the possibility of powerful AI models facilitating bio-terrorist attacks. They also developed advanced safety measures to prevent misuse, including countering AI-generated ransomware and fraudulent schemes. Anthropic's alignment team reported on emergent misalignment through reward-hacking in AI training.

---

[Web Result 1 | Ashutosh Kumar Singh's Post - LinkedIn | https://www.linkedin.com/posts/ashutosh-kumar-singh951_in-february-2025-anthropic-put-its-ai-safety-activity-7454268024674467840-5Ie8]
In February 2025, Anthropic put its AI safety system to the ultimate test — and the results were a masterclass in what real security research

---

[Web Result 2 | Detecting and countering misuse of AI: August 2025 - Anthropic | https://www.anthropic.com/news/detecting-countering-misuse-aug-2025]
We’ve [developed](https://www.anthropic.com/news/building-safeguards-for-claude) sophisticated safety and security measures to prev